![](https://miro.medium.com/v2/resize:fit:786/format:webp/1*44eDEuZBEsmG_TCAKRI3Kw@2x.png)

In [ ]:
!pip install lightning>=2.0.0
!pip install hydra-core>=1.3.2 hydra-colorlog==1.2.0
!pip install torchmetrics>=0.11.4

In [ ]:
!mkdir src

In [ ]:
%%writefile src/net.py
import torch
import torch.nn as nn
from collections import namedtuple

Output = namedtuple("Output", ["embedding", "reconstructed", "loss"])

class AutoEncoder(nn.Module):
    def __init__(self, img_size: int=28, latent_dim: int=2):
        super().__init__()
        self.img_size = img_size
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(img_size*img_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, img_size*img_size),
            nn.Sigmoid()
        )

    def forward_enc(self, x):
        # x: batch_size, channel, height, width
        x = x.flatten(1) # batch_size, channel*height*width
        return self.encoder(x)

    def forward_dec(self, z):
        # z: batch_size, latent_dim
        return self.decoder(z)

    @torch.no_grad()
    def reconstruct(self, z, shape):
        xprime = self.decoder(z)
        xprime = xprime.reshape(shape)
        return xprime

    def forward(self, x):
        # z: batch_size, latent_dim
        z = self.forward_enc(x)
        xprime = self.forward_dec(z)
        xprime = xprime.reshape(x.shape)

        error_sq = (x - xprime)**2
        loss = error_sq.mean(dim=0).sum()
        return Output(embedding=z, reconstructed=xprime, loss=loss)

if __name__ == "__main__":
    model = AutoEncoder()
    rand = torch.rand(4, 1, 28, 28)
    output = model(rand)
    print(output.embedding.shape)
    print(output.reconstructed.shape)
    print(output.loss)


In [ ]:
!python src/net.py

# Data

In [ ]:
%%writefile src/data.py
import lightning as L
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader


class MNISTDATAModule(L.LightningDataModule):
    def __init__(self, batch_size: int=64, data_dir: str="./data", img_size: int=28):
        super().__init__()
        self.save_hyperparameters()

        tensor_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])

        self.train_set = MNIST(data_dir, train=True, download=True, transform=tensor_transform)
        self.valid_set = MNIST(data_dir, train=False, download=True, transform=tensor_transform)

    def train_dataloader(self):
        return DataLoader(self.train_set, batch_size=self.hparams.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.valid_set, batch_size=self.hparams.batch_size, shuffle=False)

if __name__ == "__main__":
    dm = MNISTDATAModule()
    images, labels = next(iter(dm.train_dataloader()))
    print(images.shape, labels.shape)

In [ ]:
!python src/data.py

# Training

In [ ]:
%%writefile train.py
import os
import torch
from torchvision.utils import save_image
import lightning as L
from lightning.pytorch import seed_everything
from lightning.pytorch.callbacks import TQDMProgressBar
import hydra
from omegaconf import DictConfig
from src.net import AutoEncoder
from src.data import MNISTDATAModule
from torchmetrics import MeanMetric
from safetensors.torch import save_model


class ModelModule(L.LightningModule):
    def __init__(self, net, learning_rate: float=5e-4):
        super().__init__()
        self.save_hyperparameters(ignore=["net"])
        self.net = net
        self.val_loss = MeanMetric()

    def on_train_start(self):
        self.val_loss.reset()

    def training_step(self, batch, batch_idx):
        images, labels = batch
        bs = images.shape[0]
        output = self.net(images)
        self.log("train_loss", output.loss, prog_bar=True, on_step=True, on_epoch=False, batch_size=bs)
        return output.loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        bs = images.shape[0]
        output = self.net(images)
        self.val_loss(output.loss)

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        data = next(iter(self.trainer.val_dataloaders.dataset))
        images = data[0].unsqueeze(0) # create batch dimension
        z = self.net.forward_enc(images)
        reconstructed = self.net.reconstruct(z, images.shape)
        dir_path = "gen_sampels"
        os.makedirs(dir_path, exist_ok=True)
        file_path = os.path.join(dir_path, f"gen_iter_{self.trainer.fit_loop.epoch_loop.total_batch_idx}.png")
        save_image(reconstructed, file_path)

        self.log("valid_loss", self.val_loss.compute(), prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.net.parameters(), lr=self.hparams.learning_rate)

@hydra.main(version_base="1.3", config_path="configs", config_name="config")
def main(cfg: DictConfig) -> None:
    seed_everything(cfg.seed, workers=True)
    dm = MNISTDATAModule(**cfg.data, img_size=cfg.net.img_size)
    net = AutoEncoder(**cfg.net)
    model = ModelModule(net, **cfg.model)
    trainer = L.Trainer(
        check_val_every_n_epoch=None,
        callbacks=[TQDMProgressBar(refresh_rate=50)],
        **cfg.trainer
    )
    trainer.fit(model, dm)
    save_model(net, "model.safetensors")

if __name__=="__main__":
    main()

In [ ]:
!mkdir configs

In [ ]:
%%writefile configs/config.yaml
seed: 3407
model:
    learning_rate: 0.0005
net:
    img_size: 28
    latent_dim: 2
data:
    batch_size: 64
    data_dir: "./data"
trainer:
    val_check_interval: 2500
    max_steps: 25000
    max_epochs: -1

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir lightning_logs/

In [ ]:
!python train.py

In [ ]:
import torch
from src.net import AutoEncoder
from safetensors.torch import load_model


net = AutoEncoder(28, 2)
load_model(net, "model.safetensors")
net.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = net.to(device)

In [ ]:
from src.data import MNISTDATAModule
import matplotlib.pyplot as plt


dm = MNISTDATAModule(64, data_dir="./data", img_size=28)

generated_idx = 150
image, label = dm.valid_set[generated_idx]
images = image.unsqueeze(0).to(device)

z = net.forward_enc(images)
reconstructed = net.reconstruct(z, images.shape)
reconstructed = reconstructed.squeeze().to("cpu").detach().numpy()

plt.imshow(reconstructed, cmap="gray")
plt.title("Generated")
plt.axis("off")
plt.show()

In [ ]:
image = image.to('cpu')
plt.imshow(image.squeeze(), cmap="gray")
plt.title("Original")
plt.axis("off")
plt.show()

In [ ]:
latents = []
labels = []
for image, label in dm.valid_set:
    images = image.unsqueeze(0).to(device)
    z = net.forward_enc(images)
    latents.append(z[0])
    labels.append(label)

In [ ]:
latents = torch.stack(latents).detach()
labels = torch.tensor(labels)
print(latents.shape, labels.shape)

In [ ]:
plt.figure(figsize=(10, 8))
for i in range(10):
    mask = labels == i
    z = latents[mask]
    z = z.numpy()
    plt.scatter(z[:, 0], z[:, 1], label=i)
    plt.legend(loc='best')
plt.title("Latent Space");

In [ ]:
z = torch.tensor([latents[generated_idx, 0], latents[generated_idx, 1]])
xprime = net.reconstruct(z.unsqueeze(0), image.shape)
xprime = xprime.to('cpu').detach().numpy()
plt.imshow(xprime[0], cmap="gray")
plt.title("Generated")
plt.axis("off")
plt.show()

In [ ]:
z = torch.zeros((1, 2))
xprime = net.reconstruct(z, image.shape)
xprime = xprime.to('cpu').detach().numpy()
plt.imshow(xprime[0], cmap="gray")
plt.title("Generated")
plt.axis("off")
plt.show()

In [ ]:
rand = torch.randn(1, 2)
xprime = net.reconstruct(rand, image.shape)
xprime = xprime.to('cpu').detach().numpy()
plt.imshow(xprime[0], cmap="gray")
plt.title("Generated")
plt.axis("off")
plt.show()